Set up the complete JAXCMR repository in VS Code on an **Apple Silicon Mac** or **Windows x64 PC**, then run a small CMR simulation. uv manages Python and the repository's dependencies.

## Install VS Code and Git

Download [Visual Studio Code](https://code.visualstudio.com/download) for your computer. On Mac, move it to Applications; on Windows, run the installer. Open VS Code's Extensions view and install **Python** and **Jupyter**, both published by Microsoft.

Install Git for your operating system:

::: {.panel-tabset}

### Apple Silicon Mac

Open **Terminal → New Terminal** in VS Code and run:

```bash
xcode-select --install
```

Complete the Command Line Tools installer. If the tools are already installed, continue.

### Windows

Install [Git for Windows](https://git-scm.com/downloads/win). Keep the option that makes Git available from the command line and other software, then close and reopen VS Code.

:::

In a new VS Code terminal, run `git --version` to confirm that Git is available.

## Clone and open the repository

Open the Command Palette (**Cmd+Shift+P** on Mac; **Ctrl+Shift+P** on Windows), choose **Git: Clone**, and enter:

```text
https://github.com/githubpsyche/jaxcmr
```

Choose a parent folder, such as Documents. Git creates a `jaxcmr` folder there. Choose **Open** when cloning finishes. If you already have a checkout, use **File → Open Folder** to open it instead.

## Install Python and dependencies

Open **Terminal → New Terminal**. It should start in the `jaxcmr` folder, where `pyproject.toml` and `uv.lock` are visible in the Explorer. Enter the following commands in this terminal, not in a notebook cell.

Install [uv](https://docs.astral.sh/uv/getting-started/installation/) using the command for your operating system:

::: {.panel-tabset}

### Apple Silicon Mac

```bash
curl -LsSf https://astral.sh/uv/install.sh | sh
```

### Windows

Use a **PowerShell** terminal:

```powershell
powershell -ExecutionPolicy ByPass -c "irm https://astral.sh/uv/install.ps1 | iex"
```

:::

Close and reopen VS Code so it can find uv. Reopen the repository folder and a new terminal, then run:

```bash
uv sync
```

The repository's `.python-version` selects Python 3.12; uv downloads it if needed, creates `.venv`, and installs JAXCMR and the default development tools from `uv.lock`. JAXCMR is installed in editable mode, so imports use this checkout's source files. The first download can take several minutes; you do not need to install Python separately or activate the environment.

Specialized tools are optional: use `uv sync --group embeddings` for sentence-transformers,
`uv sync --group profiling` for line-profiler, or `uv sync --extra experimental` for the
Numba conversion helper. The normal notebook workflows need none of these. Packaging checks
use `uv sync --extra test`; see [Testing](https://github.com/githubpsyche/jaxcmr/blob/main/TESTING.md).

## Open the notebook and choose its kernel

In the Explorer, open **`guide/developer-setup.ipynb`**. This is the notebook behind this page, including the quickstart below.

Choose **Select Kernel → Select Another Kernel → Python Environments**, then select the repository's `.venv`. Some VS Code versions show **Python Environments** directly. The kernel is the Python process that runs notebook cells. Its executable is:

| Platform | Interpreter inside the repository |
|---|---|
| Apple Silicon Mac | `.venv/bin/python` |
| Windows | `.venv\Scripts\python.exe` |

If `.venv` is missing, use **Python: Select Interpreter → Enter interpreter path** from the Command Palette to select that file, then return to the kernel picker. If VS Code asks to install `ipykernel`, confirm that `uv sync` completed and that this environment is selected; the repository already includes it.

An import error usually means the wrong kernel is selected or installation did not finish. On Windows, a JAX DLL error may require the [Microsoft Visual C++ Redistributable for x64](https://learn.microsoft.com/en-us/cpp/windows/latest-supported-vc-redist); restart the kernel after installing it. See [JAX's CPU installation instructions](https://docs.jax.dev/en/latest/installation.html#pip-installation-cpu).

To return later, reopen the same folder and notebook, select the same kernel, and use **Restart Kernel**, then **Run All**. After updating the checkout, rerun the `uv sync` command first. See [VS Code's notebook instructions](https://code.visualstudio.com/docs/datascience/jupyter-notebooks) for the editor controls.

## Working with template paths

Templates find the enclosing Git project even when the kernel starts inside `templates/`
or a nested job folder. Their relative inputs and outputs use that project root. You do not
need a particular VS Code or Jupyter working-directory setting.

Set the template's `project_root` parameter when the kernel starts outside your project.
For a downloaded notebook without a Git checkout, the default is the kernel's working folder.
Absolute input and output paths remain absolute. On Windows, use forward slashes, for example
`project_root = "C:/Users/Alex/Documents/my-study"`, or a Python raw string such as
`r"C:\Users\Alex\Documents\my-study"`. See the [template path rules](templates.qmd#paths-and-working-directories).

## Quick start

This is the [earlier quickstart](https://github.com/githubpsyche/jaxcmr/blob/ae04fe1d/index.qmd), updated for the current API. Run the following Python cells in order with **Shift+Enter**. No dataset or fitted model is required.

In [1]:
#| code-summary: "1 — Import packages and check the environment"

import sys

from jax import lax, random
import jax.numpy as jnp

import jaxcmr
from jaxcmr.models.cmr import CMR
from jaxcmr.simulation import simulate_study_and_free_recall

print("Python:", sys.version.split()[0])
print("Environment:", sys.executable)
print("JAXCMR source:", jaxcmr.__file__)


Expect Python 3.12, an interpreter inside the repository's `.venv`, and a JAXCMR source path ending in `jaxcmr/jaxcmr/__init__.py` inside your checkout.

In [2]:
#| code-summary: "2 — Set the example model parameters"

params = {
    "encoding_drift_rate": 0.5,
    "start_drift_rate": 0.5,
    "recall_drift_rate": 0.5,
    "shared_support": 0.05,
    "item_support": 0.25,
    "primacy_scale": 1.0,
    "primacy_decay": 0.6,
    "learning_rate": 0.5,
    "stop_probability_scale": 0.05,
    "stop_probability_growth": 0.2,
    "choice_sensitivity": 0.6,
    "learn_after_context_update": False,
    "allow_repeated_recalls": False,
}


In [3]:
#| code-summary: "3 — Simulate study and free recall"

list_length = 5
present = jnp.arange(1, list_length + 1)
rng = random.PRNGKey(0)

model = CMR(list_length, params)
# Free recall does not use the trial argument in this shared interface.
trial = jnp.zeros_like(present)
model, recalls = simulate_study_and_free_recall(model, present, trial, rng)
print("Simulated recalls:", recalls)


Simulated recalls: [1 5 2 4 3]


The numbers identify recalled items within the five-item list; zero denotes padding after stopping. The fixed random seed makes this example repeatable.

In [4]:
#| code-summary: "4 — Inspect probabilities after studying the list"

model = CMR(list_length, params)
model = lax.fori_loop(
    0, present.size, lambda i, model: model.experience(present[i]), model
)
model = model.start_retrieving()
print(model.outcome_probabilities())


[0.05       0.23007919 0.19744956 0.17994162 0.17206511 0.17046441]


The first entry is the probability of stopping; the remaining entries correspond to items 1–5. Together they sum to one.

In [5]:
#| code-summary: "5 — Recall item 5 and inspect the updated probabilities"

model = model.retrieve(5)
print(model.outcome_probabilities())


[0.06107014 0.2645688  0.23522906 0.22138919 0.21774279 0.        ]


The probabilities change after retrieval. The final entry becomes zero because this example disables repeated recalls. Reaching these outputs confirms that the checkout imports, JAX runs, and the notebook kernel can execute the model.